# CS701 SAR benchmark — self-contained Colab training

Everything runs inside this notebook: the code is written to `/content/sar` by the cells below (no `git clone`).
Results, checkpoints and submission ZIPs are saved to Google Drive so they survive the session ending.

**Before you start**

1. **Runtime → Change runtime type → GPU: A100 (best) or L4.** Training uses bf16, which needs an
   Ampere-or-newer GPU; the free T4 is not supported. TerraMind + MoE-LoRA takes about 3 h on a fast GPU,
   longer on L4.
2. **Upload the course data to Google Drive once:** put `train.tar.gz` and `val.tar.gz` (the two files in the
   repo's `data/` folder) in `My Drive/sar-benchmark/data/`.
3. **Run all cells top to bottom.** Training cannot resume, so keep the session alive until it finishes.

## PREPROC CHANGELOG

Every change is marked `PREPROC[...]` (Ctrl+F). All flags default OFF, which reproduces the original pipeline exactly. Priority: P1 gamma first, then P2 gain; P3 medium; P4 per-image norm and P5 despeckle are low-priority experiments. Unverified points are marked `TODO(PREPROC)`.

| cell | what changed | flag |
|---|---|---|
| 4 | Settings: `GAMMA`, `GAIN`, `NORM`, `DESPECKLE`, `TM_AUX`, `TAG`; builds `PREPROC_ARGS` and adds `PREPROC_SUFFIX` to `VARIANT` (so `RUN_NAME`, results, checkpoints and ZIP name all differ per variant) | all |
| 14 | `data.py`: `Gamma`, `RandomGain`, `PerImageNorm`, `EnhancedLee`, `NORM_STATS`/`get_norm_stats`, `build_transform`; `SARMultiTask` uses `build_transform` (new kwargs `despeckle`, `gamma`, `gain`, `norm`) | P1, P2, P4, P5 |
| 15 | `backbones.py`: `aux` argument; TerraMind second S1GRD channel copy or 3x3 mean | `--tm-aux` (P3) |
| 20 | `train.py`: five new flags plus validation; passes them to `SARMultiTask` and `build_backbone`; saved in `metrics.json` through `vars(args)` | all |
| 21 | `make_submission.py`: `inference_transform` calls `build_transform` with the args from `metrics.json` (gain always off); `aux` passed to `build_backbone` | all |
| 22 | New `compute_norm_stats.py` | P1, P5 |
| 30 | New section 6b: restore `norm_stats.json` from Drive; compute stats for the current settings | P1, P5 |
| 35 | New section 7b: parity, train/inference parity, smoke runs, submission smoke | verification |
| 41 | Train cell: `{PREPROC_ARGS}` appended to the `train.py` command | all |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Settings

In [ ]:
# --- Experiment (same options as train.py) ---
BACKBONE = 'terramind'      # 'vit' or 'terramind'
INIT = 'pretrained'         # 'pretrained' or 'scratch' (scratch needs ADAPT='full'; over the 40M budget)
ADAPT = 'moelora'           # 'lora' or 'moelora' ('full' is over the 40M trainable budget)
SEED = 0
EPOCHS = 24
BATCH_SIZE = 32
LR = 1e-4
EVAL_EVERY = 1
EXTRA_ARGS = ''             # e.g. '--anchor-profile wide --class-sampling sqrt-balanced'

# >>> PREPROC[settings] optional preprocessing stages; these defaults reproduce the original pipeline exactly
GAMMA = 1.0                 # P1 (run first): pixel**GAMMA on [0, 1], e.g. 0.5; needs norm stats (section 6b)
GAIN = False                # P2: random global gain U(0.85, 1.18), training only
NORM = 'dataset'            # P4: 'dataset' or 'per_image' (EXPERIMENT, expected to hurt)
DESPECKLE = 'none'          # P5: 'none' or 'lee3' (low priority); needs norm stats (section 6b)
TM_AUX = 'copy'             # P3: TerraMind second channel 'copy' or 'mean3' (BACKBONE = 'terramind' only)
TAG = ''                    # optional free-text suffix for the run name
PREPROC_ARGS, PREPROC_SUFFIX = '', ''   # only the non-default flags; the suffix keeps variants apart on Drive
if GAMMA != 1.0:
    PREPROC_ARGS += f' --gamma {GAMMA:g}'; PREPROC_SUFFIX += f'_g{GAMMA:g}'
if GAIN:
    PREPROC_ARGS += ' --gain'; PREPROC_SUFFIX += '_gain'
if NORM != 'dataset':
    PREPROC_ARGS += f' --norm {NORM}'; PREPROC_SUFFIX += '_perimg'
if DESPECKLE != 'none':
    PREPROC_ARGS += f' --despeckle {DESPECKLE}'; PREPROC_SUFFIX += f'_{DESPECKLE}'
if TM_AUX != 'copy':
    PREPROC_ARGS += f' --tm-aux {TM_AUX}'; PREPROC_SUFFIX += f'_{TM_AUX}'
if TAG:
    PREPROC_SUFFIX += f'_{TAG}'
# <<< PREPROC[settings]

# Drive folders mirror the repo layout, so they can be copied straight into the git repo:
#   results/<GROUP>/<VARIANT>/<backbone>_<init>_<adapt>/seed_<SEED>/   metrics, holdout predictions + annotations
#   checkpoints/<GROUP>/<VARIANT>/<backbone>_<init>_<adapt>/seed_<SEED>/   model.pt, metrics.json
#   submissions/<DATE>/tests-no-submit/<ZIP_NAME>.zip   (+ .metrics.json, .gpu.txt)
DATE = '161026'                      # DDMMYY, same as the repo's submissions/ date folders
GROUP = f'colab-{DATE}'
VARIANT = f'{BACKBONE}_{EPOCHS}ep{PREPROC_SUFFIX}'   # add a suffix if you change EXTRA_ARGS, so runs never overwrite each other  # <<< PREPROC[settings] (was: VARIANT = f'{BACKBONE}_{EPOCHS}ep')
RUN_NAME = f'{GROUP}/{VARIANT}'
ZIP_NAME = f'{VARIANT}_seed{SEED}'
print('RUN_NAME', RUN_NAME, '| PREPROC_ARGS', PREPROC_ARGS or '(none)')  # <<< PREPROC[settings] (was: none)

# --- Google Drive locations ---
DRIVE_ROOT = '/content/drive/MyDrive/sar-benchmark'
DRIVE_DATA_DIR = f'{DRIVE_ROOT}/data'   # must contain train.tar.gz and val.tar.gz

## 2. Check the GPU

In [ ]:
import subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
cap = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
if not cap:
    raise RuntimeError('No GPU found. Runtime -> Change runtime type -> select a GPU (A100 or L4).')
if float(cap.splitlines()[0]) < 8.0:
    print(f'WARNING: compute capability {cap} has no native bf16 (e.g. T4). Switch to an A100 or L4 runtime.')
else:
    print(f'OK: compute capability {cap} supports bf16.')

## 3. Mount Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import os
for sub in ('runs', 'results', 'checkpoints', 'submissions', 'hf_cache'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)
# Cache pretrained weights (ViT 0.4 GB, TerraMind 1.5 GB) on Drive so later sessions skip the download
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
for name in ('train.tar.gz', 'val.tar.gz'):
    path = f'{DRIVE_DATA_DIR}/{name}'
    print(('found   ' if os.path.exists(path) else 'MISSING ') + path)

## 4. Write the code

These cells write the training package, `train.py`, `make_submission.py` and the pinned requirements.

In [ ]:
import os
os.makedirs('/content/sar/sarbench', exist_ok=True)
%cd /content/sar

In [ ]:
%%writefile /content/sar/requirements.txt
--extra-index-url https://download.pytorch.org/whl/cu128  # CUDA 12.8 torch/torchvision wheels (Blackwell GPUs)
torch==2.9.0
torchvision==0.24.0
timm==1.0.24
terratorch==1.2.7
torchgeo==0.9.0  # terratorch 1.2.7 imports torchgeo.trainers.utils, removed in torchgeo 0.10
pycocotools==2.0.11
pytest==9.1.1

In [ ]:
!pip uninstall -y torchaudio

In [ ]:
%%writefile /content/sar/sarbench/__init__.py
"""The model and data code behind train.py: data loading, the two backbones, LoRA / MoE-LoRA adapters, the
multi-task model and the metrics."""

In [ ]:
%%writefile /content/sar/sarbench/data.py
"""Labeled course training data: every image has one class label and >= 1 box.
Loads images and boxes at a fixed input size and maps predicted boxes back to original pixels."""
import csv
import json
from collections import defaultdict
from pathlib import Path

import torch
# >>> PREPROC[P5-despeckle] avg_pool2d for the enhanced Lee filter
import torch.nn.functional as F
# <<< PREPROC[P5-despeckle]
from PIL import Image
from torch.utils.data import Dataset
from torchvision import tv_tensors
from torchvision.ops import box_convert
from torchvision.transforms import v2

# Mean and std of all pixels of the 9,392 train images, read as gray, resized to
# 512 x 512 and scaled to [0, 1] (computed once).
MEAN, STD = 0.2526, 0.2133


def padded_size(orig_size, size=512):
    h, w = orig_size
    scale = size / max(h, w)
    return max(1, round(h * scale)), max(1, round(w * scale))


class ResizePad:
    """Resize without stretching, then pad on the right and bottom (also transforms boxes)."""

    def __init__(self, size=512):
        self.size = size

    def __call__(self, image, boxes=None):
        from torchvision.transforms.v2 import functional as TF
        h, w = padded_size(image.shape[-2:], self.size)
        image = TF.pad(TF.resize(image, [h, w]), [0, 0, self.size - w, self.size - h])
        if boxes is None:
            return image
        boxes = TF.pad(TF.resize(boxes, [h, w]), [0, 0, self.size - w, self.size - h])
        return image, boxes


# >>> PREPROC[P1-gamma] power-law stretch of [0, 1] pixels (gamma < 1 brightens the dark mid-range)
class Gamma:
    def __init__(self, g): self.g = g
    def __call__(self, image, boxes=None):
        image = tv_tensors.Image(image.clamp(0, 1) ** self.g)
        return image if boxes is None else (image, boxes)
# <<< PREPROC[P1-gamma]


# >>> PREPROC[P2-gain] random global brightness gain, training only (never used at inference)
class RandomGain:
    def __init__(self, lo=0.85, hi=1.18): self.lo, self.hi = lo, hi
    def __call__(self, image, boxes=None):
        g = float(torch.empty(1).uniform_(self.lo, self.hi))
        image = tv_tensors.Image((image * g).clamp(0, 1))
        return image if boxes is None else (image, boxes)
# <<< PREPROC[P2-gain]


# >>> PREPROC[P4-pernorm] EXPERIMENT: expected to remove a real brightness class cue
class PerImageNorm:
    def __call__(self, image, boxes=None):
        m, s = image.mean(), image.std().clamp_min(0.02)   # floor stops near-black images exploding
        image = tv_tensors.Image((image - m) / s)
        return image if boxes is None else (image, boxes)
# <<< PREPROC[P4-pernorm]


# >>> PREPROC[P5-despeckle] LOW PRIORITY. sigma_u=0.1 and 3x3 are calibrated to this data
# (calm-window CV about 0.09). A larger sigma_u (e.g. 0.5) would flatten the whole image and erase small targets.
class EnhancedLee:
    def __init__(self, win=3, sigma_u=0.1): self.win, self.sigma_u = win, sigma_u
    def __call__(self, image, boxes=None):
        x = image.float() / 255.0 if image.dtype == torch.uint8 else image.float()
        p = self.win // 2
        m = F.avg_pool2d(x[None], self.win, 1, p, count_include_pad=False)[0]
        v = (F.avg_pool2d((x * x)[None], self.win, 1, p, count_include_pad=False)[0] - m * m).clamp_min(0)
        cu2 = self.sigma_u ** 2
        cg = v.sqrt() / m.clamp_min(1e-3)
        k = ((v - m * m * cu2) / (v * (1 + cu2) + 1e-12)).clamp(0, 1)
        out = m + k * (x - m)
        out = torch.where(cg > 3 ** 0.5 * self.sigma_u, x, out)   # strong scatterers left untouched
        out = torch.where(cg < self.sigma_u, m, out)              # homogeneous: local mean
        image = tv_tensors.Image(out.clamp(0, 1))
        return image if boxes is None else (image, boxes)
# <<< PREPROC[P5-despeckle]


# >>> PREPROC[P1-gamma] normalisation constants depend on gamma and despeckle
NORM_STATS = {'1|none': (0.2526, 0.2133)}      # the original constants

def get_norm_stats(gamma, despeckle):
    import json, os
    # SARBENCH_NORM_STATS: alternative stats file, used only by the smoke-test cells so they never touch the real one
    path = os.environ.get('SARBENCH_NORM_STATS') or os.path.join(os.path.dirname(__file__), '..', 'norm_stats.json')
    stats = dict(NORM_STATS)
    if os.path.exists(path):
        stats.update({k: tuple(v) for k, v in json.load(open(path)).items()})
    key = f'{gamma:g}|{despeckle}'
    if key not in stats:
        raise KeyError(f'No normalisation stats for {key}. Run the "norm stats" notebook cell for these settings first.')
    return stats[key]
# <<< PREPROC[P1-gamma]


# >>> PREPROC[build_transform] the one preprocessing pipeline shared by training, validation and inference
def build_transform(size=512, geometry='stretch', train=False, despeckle='none', gamma=1.0, gain=False, norm='dataset'):
    """With all defaults this is exactly the original SARMultiTask pipeline (verified by the PREPROC parity cell)."""
    if despeckle not in ('none', 'lee3'):
        raise ValueError(f'unknown despeckle {despeckle!r}')
    if norm not in ('dataset', 'per_image', 'none'):
        raise ValueError(f'unknown norm {norm!r}')
    steps = [v2.ToImage(), v2.ConvertBoundingBoxFormat('XYXY')]  # COCO [x, y, w, h] -> [x1, y1, x2, y2]
    if despeckle == 'lee3':
        steps.append(EnhancedLee(win=3, sigma_u=0.1))  # native resolution, before resizing
    steps.append(ResizePad(size) if geometry == 'pad' else v2.Resize((size, size)))
    if train:
        steps += [v2.RandomHorizontalFlip(), v2.RandomVerticalFlip()]  # overhead imagery has no canonical "up"
    steps.append(v2.ToDtype(torch.float32, scale=True))
    if train and gain:
        steps.append(RandomGain())
    if gamma != 1.0:
        steps.append(Gamma(gamma))
    if norm == 'dataset':
        m, s = get_norm_stats(gamma, despeckle)
        steps.append(v2.Normalize([m], [s]))
    elif norm == 'per_image':
        steps.append(PerImageNorm())
    return v2.Compose(steps)
# <<< PREPROC[build_transform]


class SARMultiTask(Dataset):
    """Images from train/labels.csv with boxes from train/instances.json under the data root.
    An item is (image [1, size, size], label 0..8, target), target = {'boxes': XYXY on the resized
    image, 'labels': 1..9, 'image_id': COCO id, 'orig_size': (h, w)}."""

    def __init__(self, root, train=False, size=512, limit=None, indices=None, geometry='stretch',
                 despeckle='none', gamma=1.0, gain=False, norm='dataset'):  # <<< PREPROC[build_transform] (was: def __init__(self, root, train=False, size=512, limit=None, indices=None, geometry='stretch'):)
        self.root = Path(root)
        self.geometry = geometry
        self.image_root = self.root / 'train'
        with open(self.image_root / 'labels.csv', newline='') as f:
            self.rows = list(csv.DictReader(f))[:limit]
        if indices is not None:
            self.rows = [self.rows[i] for i in indices]
        coco = json.loads((self.image_root / 'instances.json').read_text())
        self.coco_images = {img['file_name']: img for img in coco['images']}
        self.annotations = defaultdict(list)  # COCO image id -> its boxes
        for ann in coco['annotations']:
            self.annotations[ann['image_id']].append(ann)
        # >>> PREPROC[build_transform] replaces the inline v2.Compose (ToImage, XYXY, Resize/ResizePad, flips,
        # ToDtype, Normalize(MEAN, STD)); identical with default flags, and shared with make_submission.py
        self.transform = build_transform(size, geometry, train, despeckle=despeckle, gamma=gamma, gain=gain, norm=norm)
        # <<< PREPROC[build_transform]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        info = self.coco_images[row['file_name']]
        anns = self.annotations[info['id']]
        orig_size = (info['height'], info['width'])
        boxes = tv_tensors.BoundingBoxes([a['bbox'] for a in anns], format='XYWH', canvas_size=orig_size,
                                         dtype=torch.float32)  # integer boxes would be truncated by Resize
        image = Image.open(self.image_root / row['file_name']).convert('L')  # PIL: torchvision.io cannot read .bmp
        image, boxes = self.transform(image, boxes)
        target = {'boxes': boxes, 'labels': torch.tensor([a['category_id'] for a in anns]),
                  'image_id': info['id'], 'orig_size': orig_size, 'geometry': self.geometry}
        return image, int(row['label']), target


def collate(batch):
    """Stack images and labels; targets stay a list because images have different numbers of boxes."""
    images, labels, targets = zip(*batch)
    return torch.stack(images), torch.tensor(labels), list(targets)


def to_original_xywh(boxes_xyxy, orig_size, size=512, geometry='stretch'):
    """Undo the resize: XYXY boxes on the size x size input -> COCO [x, y, w, h] in original pixels."""
    h, w = orig_size
    rh, rw = padded_size(orig_size, size) if geometry == 'pad' else (size, size)
    boxes = boxes_xyxy * boxes_xyxy.new_tensor([w / rw, h / rh, w / rw, h / rh])
    if geometry == 'pad':
        boxes = torch.stack([boxes[:, 0].clamp(0, w), boxes[:, 1].clamp(0, h),
                             boxes[:, 2].clamp(0, w), boxes[:, 3].clamp(0, h)], dim=1)
    return box_convert(boxes, 'xyxy', 'xywh')

In [ ]:
%%writefile /content/sar/sarbench/backbones.py
"""The two ViT-B/16 backbones, built for 512 px input: images [B, 1, 512, 512] -> patch tokens [B, 1024, 768]."""
import timm
# >>> PREPROC[P3-tm-aux] avg_pool2d for the 3x3 local-mean channel
import torch, torch.nn.functional as F
# <<< PREPROC[P3-tm-aux]
from terratorch.registry import BACKBONE_REGISTRY
from torch import nn


class ViT(nn.Module):
    """timm ViT-B/16 pretrained on ImageNet-21k."""

    def __init__(self, pretrained: bool, aux='copy'):  # <<< PREPROC[P3-tm-aux] aux ignored by ViT (was: def __init__(self, pretrained: bool):)
        super().__init__()
        # in_chans=1: timm sums the RGB patch-embedding weights; img_size=512: it resamples the position embedding
        self.vit = timm.create_model('vit_base_patch16_224.augreg_in21k', pretrained=pretrained,
                                     img_size=512, in_chans=1, num_classes=0)
        self.embed_dim = 768

    @property
    def blocks(self):
        return self.vit.blocks

    def forward(self, x):
        tokens = self.vit.forward_features(x)  # [B, 1 + N, 768], final norm applied
        return tokens[:, self.vit.num_prefix_tokens:]  # drop the CLS token


class TerraMind(nn.Module):
    """TerraMind-1.0-base (IBM/ESA) through its Sentinel-1 GRD input."""

    def __init__(self, pretrained: bool, aux='copy'):  # <<< PREPROC[P3-tm-aux] (was: def __init__(self, pretrained: bool):)
        super().__init__()
        self.aux = aux  # <<< PREPROC[P3-tm-aux] (was: none)
        self.terramind = BACKBONE_REGISTRY.build('terratorch_terramind_v1_base', pretrained=pretrained,
                                                 modalities=['S1GRD'])
        self.embed_dim = 768

    @property
    def blocks(self):
        return self.terramind.encoder

    def forward(self, x):
        # TODO(PREPROC): unverified which TerraMind S1GRD channel is VV/VH, and whether TerraMind expects dB-scaled input. Check its documentation before trusting this ablation.
        # >>> PREPROC[P3-tm-aux] second S1GRD channel: raw copy (original) or 3x3 local mean of the grey image
        if self.aux == 'mean3':
            x2 = torch.cat([x, F.avg_pool2d(x, 3, 1, 1, count_include_pad=False)], dim=1)
        else:
            x2 = x.repeat(1, 2, 1, 1)
        per_block = self.terramind({'S1GRD': x2})
        # <<< PREPROC[P3-tm-aux] (was: per_block = self.terramind({'S1GRD': x.repeat(1, 2, 1, 1)})  # gray image as both VV and VH)
        return per_block[-1]  # last block's tokens [B, N, 768]; terratorch already applied encoder_norm


def build_backbone(name, pretrained, aux='copy'):  # <<< PREPROC[P3-tm-aux] (was: def build_backbone(name, pretrained):)
    """name: 'vit' or 'terramind'. pretrained=False builds the same architecture with random weights."""
    return {'vit': ViT, 'terramind': TerraMind}[name](pretrained, aux=aux)  # <<< PREPROC[P3-tm-aux] (was: ...[name](pretrained))

In [ ]:
%%writefile /content/sar/sarbench/adapters.py
"""LoRA and MoE-LoRA adapters for the attention layers of a frozen backbone."""
from torch import nn


class LoRA(nn.Module):
    """y = W x + (alpha / r) * B A x, where W is the wrapped layer (frozen by add_adapters)."""

    def __init__(self, base: nn.Linear, rank=16, alpha=32):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.in_features, rank, bias=False)  # A
        self.up = nn.Linear(rank, base.out_features, bias=False)  # B
        nn.init.zeros_(self.up.weight)  # B = 0: training starts from exactly the pretrained layer
        self.scale = alpha / rank

    def forward(self, x):
        return self.base(x) + self.scale * self.up(self.down(x))


class MoELoRA(nn.Module):
    """y = W x + (alpha / r) * sum_e g_e(x) B_e A_e x, with gates g(x) = softmax(router[task](x)) per token.

    The experts' A_e are stacked in `down` and their B_e side by side in `up`, so gating each
    expert's r hidden units and applying `up` sums the experts.
    """

    def __init__(self, base: nn.Linear, experts=4, rank=4, alpha=8, tasks=2):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.in_features, experts * rank, bias=False)  # A_1 ... A_E
        self.up = nn.Linear(experts * rank, base.out_features, bias=False)  # B_1 ... B_E
        nn.init.zeros_(self.up.weight)
        self.routers = nn.ModuleList(nn.Linear(base.in_features, experts, bias=False) for _ in range(tasks))
        self.rank = rank
        self.scale = alpha / rank
        self.task = 0  # which router to use; see set_task

    def forward(self, x):
        gates = self.routers[self.task](x).softmax(dim=-1)
        gates = gates.repeat_interleave(self.rank, dim=-1)  # expert e's gate on each of its r hidden units
        return self.base(x) + self.scale * self.up(self.down(x) * gates)


def add_adapters(backbone, kind):
    """Freeze the backbone, then wrap attn.qkv and attn.proj of every block (kind: 'lora' or 'moelora')."""
    adapter = {'lora': LoRA, 'moelora': MoELoRA}[kind]
    backbone.requires_grad_(False)
    for blk in backbone.blocks:
        blk.attn.qkv = adapter(blk.attn.qkv)
        blk.attn.proj = adapter(blk.attn.proj)


def unfreeze_last_blocks(backbone, count):
    """Allow the last `count` encoder blocks to train alongside the adapters."""
    if count < 0 or count > len(backbone.blocks):
        raise ValueError(f'Cannot unfreeze {count} blocks from a {len(backbone.blocks)}-block backbone.')
    if count:
        for block in backbone.blocks[-count:]:
            block.requires_grad_(True)


def set_task(module, task):
    """Route every MoELoRA inside `module` through the router of `task` (0 = classification, 1 = detection)."""
    for layer in module.modules():
        if isinstance(layer, MoELoRA):
            layer.task = task

In [ ]:
%%writefile /content/sar/sarbench/model.py
"""The multi-task model: one shared ViT backbone feeds a classification head and a Faster R-CNN detector."""
from collections import OrderedDict

import torch
import torch.nn.functional as F
from torch import nn
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor, TwoMLPHead
from torchvision.models.detection.image_list import ImageList
from torchvision.models.detection.roi_heads import RoIHeads
from torchvision.models.detection.rpn import RegionProposalNetwork, RPNHead
from torchvision.ops import MultiScaleRoIAlign

from .adapters import set_task


class SimpleFeaturePyramid(nn.Module):
    """ViTDet neck (Li et al., 2022): the single stride-16 ViT map -> 256-channel maps at strides 4, 8, 16, 32, 64."""

    def __init__(self, dim=768, channels=256):
        super().__init__()
        self.rescale = nn.ModuleList([
            nn.Sequential(nn.ConvTranspose2d(dim, dim // 2, 2, stride=2), nn.GroupNorm(32, dim // 2), nn.GELU(),
                          nn.ConvTranspose2d(dim // 2, dim // 4, 2, stride=2)),  # stride 4
            nn.ConvTranspose2d(dim, dim // 2, 2, stride=2),  # stride 8
            nn.Identity(),  # stride 16
            nn.MaxPool2d(2),  # stride 32
        ])
        self.project = nn.ModuleList(
            nn.Sequential(nn.Conv2d(c, channels, 1, bias=False), nn.GroupNorm(32, channels),
                          nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.GroupNorm(32, channels))
            for c in (dim // 4, dim // 2, dim, dim))

    def forward(self, x):
        levels = [project(rescale(x)) for rescale, project in zip(self.rescale, self.project)]
        levels.append(F.max_pool2d(levels[-1], 1, stride=2))  # stride 64, as torchvision's FPN LastLevelMaxPool
        return OrderedDict(zip(['0', '1', '2', '3', 'pool'], levels))


class MultiTaskModel(nn.Module):
    """backbone tokens -> classification logits (mean token -> LayerNorm -> Linear) and Faster R-CNN detections."""

    def __init__(self, backbone, task_routing: bool, num_classes=9, anchor_profile='default'):
        super().__init__()
        self.backbone = backbone
        self.task_routing = task_routing  # MoE-LoRA: the backbone runs once per task, with that task's routers
        self.cls_head = nn.Sequential(nn.LayerNorm(backbone.embed_dim), nn.Linear(backbone.embed_dim, num_classes))
        self.neck = SimpleFeaturePyramid(backbone.embed_dim)
        # Small objects (median ~16 px at 512 px, 99 % below ~240 px): two anchor scales per octave, from 8 px at
        # stride 4 up to 181 px at stride 64, times 3 aspect ratios = 6 anchors per location.
        sizes = ((8, 11), (16, 23), (32, 45), (64, 91), (128, 181))
        ratios = (0.5, 1.0, 2.0)
        if anchor_profile == 'wide':
            # Isolate shape coverage: preserve scales, add elongated anchors.
            ratios = (0.2, 0.5, 1.0, 2.0, 5.0)
        elif anchor_profile == 'large':
            # Isolate scale coverage: extend the coarsest level to larger objects.
            sizes = (*sizes[:4], (181, 256))
        elif anchor_profile != 'default':
            raise ValueError(f'Unknown anchor profile: {anchor_profile}')
        anchors = AnchorGenerator(sizes, (ratios,) * 5)
        # Everything else is torchvision's Faster R-CNN default.
        self.rpn = RegionProposalNetwork(
            anchors, RPNHead(256, len(sizes[0]) * len(ratios)), fg_iou_thresh=0.7, bg_iou_thresh=0.3, batch_size_per_image=256,
            positive_fraction=0.5, pre_nms_top_n=dict(training=2000, testing=1000),
            post_nms_top_n=dict(training=2000, testing=1000), nms_thresh=0.7)
        self.roi_heads = RoIHeads(
            MultiScaleRoIAlign(['0', '1', '2', '3'], output_size=7, sampling_ratio=2),
            TwoMLPHead(256 * 7 * 7, 1024), FastRCNNPredictor(1024, num_classes + 1),  # + 1: background
            fg_iou_thresh=0.5, bg_iou_thresh=0.5, batch_size_per_image=512, positive_fraction=0.25,
            bbox_reg_weights=None, score_thresh=0.05, nms_thresh=0.5, detections_per_img=100)

    def forward(self, images, targets=None):
        """images [B, 1, H, W], targets: per-image {'boxes' XYXY, 'labels' 1..9} (training only) ->
        (logits [B, num_classes], per-image detections (eval mode), the four Faster R-CNN losses (training mode))."""
        set_task(self.backbone, 0)  # 0 = classification, 1 = detection; a no-op without MoE-LoRA
        tokens = self.backbone(images)
        logits = self.cls_head(tokens.mean(dim=1))
        if self.task_routing:
            set_task(self.backbone, 1)
            tokens = self.backbone(images)
        h, w = images.shape[-2] // 16, images.shape[-1] // 16  # one token per 16 x 16 patch
        features = self.neck(tokens.transpose(1, 2).unflatten(2, (h, w)))  # tokens as a [B, 768, h, w] map
        # Precision rule: the box coder casts anchors to the dtype of the regression output, so in bf16 every
        # coordinate near 512 would snap to a 2 px grid. RPN and RoIHeads therefore run in fp32.
        with torch.autocast(images.device.type, enabled=False):
            features = {name: f.float() for name, f in features.items()}
            image_list = ImageList(images, [tuple(images.shape[-2:])] * len(images))
            proposals, rpn_losses = self.rpn(image_list, features, targets)
            detections, roi_losses = self.roi_heads(features, proposals, image_list.image_sizes, targets)
        return logits, detections, {**rpn_losses, **roi_losses}

In [ ]:
%%writefile /content/sar/sarbench/optim.py
"""Optimizer groups for adapters, task heads, and optional unfrozen backbone weights."""


def parameter_groups(model, backbone_lr, head_lr, unfrozen_lr=None):
    """Keep heads in group 1, and optionally give original backbone weights their own LR."""
    backbone = [(name, p) for name, p in model.backbone.named_parameters() if p.requires_grad]
    heads = [p for name, p in model.named_parameters() if not name.startswith('backbone.') and p.requires_grad]
    if unfrozen_lr is None:
        return [{'params': [p for _, p in backbone], 'lr': backbone_lr, 'name': 'backbone'},
                {'params': heads, 'lr': head_lr, 'name': 'heads'}]
    adapters, original = [], []
    for name, parameter in backbone:
        target = adapters if any(part in name for part in ('.down.', '.up.', '.routers.')) else original
        target.append(parameter)
    return [{'params': adapters, 'lr': backbone_lr, 'name': 'adapters'},
            {'params': heads, 'lr': head_lr, 'name': 'heads'},
            {'params': original, 'lr': unfrozen_lr, 'name': 'unfrozen_backbone'}]

In [ ]:
%%writefile /content/sar/sarbench/metrics.py
"""Evaluation metrics: classification scores from a confusion matrix, and COCO box AP via pycocotools."""
import contextlib
import io

import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def classification_metrics(y_true, y_pred, num_classes=9):
    """Accuracy, macro-F1, balanced accuracy (mean per-class recall) and the confusion matrix
    (rows = truth, columns = prediction). Every class must occur in y_true, as in every split."""
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    tp, support, predicted = np.diag(cm), cm.sum(1), cm.sum(0)
    recall = tp / support
    f1 = 2 * tp / (support + predicted)  # = 2 TP / (2 TP + FN + FP), defined for never-predicted classes
    return {
        'accuracy': float(tp.sum() / cm.sum()),
        'macro_f1': float(f1.mean()),
        'balanced_accuracy': float(recall.mean()),
        'confusion_matrix': cm.tolist(),
    }


def detection_metrics(gt_json, predictions, image_ids):
    """COCO box AP of `predictions` (dicts with image_id, category_id, bbox [x, y, w, h] in original
    pixels, score) against the ground truth in `gt_json`, evaluated on the images `image_ids` only."""
    with contextlib.redirect_stdout(io.StringIO()):  # silence pycocotools; the caller prints its own summary
        gt = COCO(gt_json)
        # loadRes adds keys to the dicts it is given, so it gets copies; it cannot load an empty list
        dt = gt.loadRes([dict(p) for p in predictions]) if predictions else COCO()
        ev = COCOeval(gt, dt, 'bbox')
        ev.params.imgIds = image_ids
        ev.evaluate()
        ev.accumulate()
        ev.summarize()
    precision = ev.eval['precision'][:, :, :, 0, -1]  # [IoU threshold, recall, class] at area "all", 100 dets
    return {
        'mAP': float(ev.stats[0]),
        'AP50': float(ev.stats[1]),
        'AP75': float(ev.stats[2]),
        'per_class_AP': {gt.cats[c]['name']: float(precision[:, :, k].mean())  # -1: no boxes of that class
                         for k, c in enumerate(ev.params.catIds)},
    }

In [ ]:
%%writefile /content/sar/train.py
"""Train classification and detection jointly, then score a holdout of labeled training images.
Example: python train.py --data data/course-data --backbone vit --init pretrained --adapt lora"""
import argparse
import json
import math
import shutil
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler

from sarbench.adapters import add_adapters, unfreeze_last_blocks
from sarbench.backbones import build_backbone
from sarbench.data import SARMultiTask, collate, to_original_xywh
from sarbench.metrics import classification_metrics, detection_metrics
from sarbench.model import MultiTaskModel
from sarbench.optim import parameter_groups

# Validation reference: mean ViT + LoRA scores over seeds 0, 1 and 2.
VAL_REFERENCE_F1 = 0.9331
VAL_REFERENCE_MAP = 0.2613


def parse_args():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--data', type=Path, default=Path(__file__).resolve().parents[1] / 'dataset' / 'SARFact-Course-20K')
    p.add_argument('--backbone', required=True, choices=['vit', 'terramind'])
    p.add_argument('--init', required=True, choices=['pretrained', 'scratch'])
    p.add_argument('--adapt', required=True, choices=['full', 'lora', 'moelora'])
    p.add_argument('--unfreeze-last', type=int, default=0,
                   help='also train the last N encoder blocks with pretrained LoRA/MoE-LoRA (default: 0)')
    p.add_argument('--unfrozen-lr', type=float, default=2e-5,
                   help='learning rate for original weights enabled by --unfreeze-last')
    p.add_argument('--epochs', type=int, default=24)
    p.add_argument('--batch-size', type=int, default=16)
    p.add_argument('--lr', type=float, default=1e-4)
    p.add_argument('--backbone-lr', type=float, help='for the trainable backbone parameters (default: --lr)')
    p.add_argument('--weight-decay', type=float, default=0.05)
    p.add_argument('--eval-every', type=int, default=4, help='validate every N epochs, and after the last one')
    p.add_argument('--workers', type=int, default=8)
    p.add_argument('--limit', type=int, help='use the first N labeled training images before splitting (quick checks)')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--anchor-profile', choices=['default', 'wide', 'large'], default='default')
    p.add_argument('--geometry', choices=['stretch', 'pad'], default='stretch')
    p.add_argument('--class-sampling', choices=['uniform', 'sqrt-balanced'], default='uniform')
    p.add_argument('--box-loss-weight', type=float, default=1.0,
                   help='multiplier for RoI box regression loss; classification and RPN losses unchanged')
    # >>> PREPROC[flags] optional preprocessing stages P1-P5; all defaults reproduce the original pipeline
    p.add_argument('--gamma', type=float, default=1.0, help='P1: pixel**gamma on [0, 1] before Normalize')
    p.add_argument('--gain', action='store_true', help='P2: random global gain U(0.85, 1.18), training only')
    p.add_argument('--norm', choices=['dataset', 'per_image'], default='dataset',
                   help='P4: dataset mean/std (default) or per-image normalisation (EXPERIMENT)')
    p.add_argument('--despeckle', choices=['none', 'lee3'], default='none',
                   help='P5: 3x3 enhanced Lee filter at native resolution (low priority)')
    p.add_argument('--tm-aux', choices=['copy', 'mean3'], default='copy',
                   help='P3: TerraMind second S1GRD channel: copy of the grey image or its 3x3 local mean')
    # <<< PREPROC[flags]
    p.add_argument('--out', type=Path, default=Path('runs'))
    p.add_argument('--results', type=Path, default=Path('results'),
                   help='parent folder for saved JSON results, organized by configuration and seed')
    p.add_argument('--checkpoints', type=Path, default=Path('checkpoints'),
                   help='parent folder for preserved model weights and configuration, organized by seed')
    args = p.parse_args()
    if not math.isfinite(args.box_loss_weight) or args.box_loss_weight <= 0:
        p.error('--box-loss-weight must be finite and positive')
    if args.init == 'scratch' and args.adapt != 'full':
        p.error('--init scratch is only valid with --adapt full')
    if args.unfreeze_last < 0 or args.unfreeze_last > 12:
        p.error('--unfreeze-last must be between 0 and 12')
    if args.unfreeze_last and (args.init != 'pretrained' or args.adapt == 'full'):
        p.error('--unfreeze-last requires pretrained initialization and lora or moelora adaptation')
    # >>> PREPROC[flags] validation
    if args.tm_aux != 'copy' and args.backbone != 'terramind':
        p.error('--tm-aux only applies to --backbone terramind')
    if not math.isfinite(args.gamma) or args.gamma <= 0:
        p.error('--gamma must be finite and positive')
    # <<< PREPROC[flags]
    if args.backbone_lr is None:
        args.backbone_lr = args.lr
    return args


def log(message, file):
    """Print to the console and to the run's log.txt."""
    print(message, flush=True)
    print(message, file=file, flush=True)


def make_loader(args, indices, train):
    """Build a loader over selected rows from the labeled course training set."""
    # >>> PREPROC[flags] (was: dataset = SARMultiTask(args.data, train=train, limit=args.limit, indices=indices, geometry=args.geometry))
    dataset = SARMultiTask(args.data, train=train, limit=args.limit, indices=indices, geometry=args.geometry,
                           despeckle=args.despeckle, gamma=args.gamma, gain=args.gain and train, norm=args.norm)
    # <<< PREPROC[flags]
    sampler = None
    if train and args.class_sampling == 'sqrt-balanced':
        labels = torch.tensor([int(row['label']) for row in dataset.rows])
        counts = torch.bincount(labels, minlength=9).clamp_min(1)
        weights = counts[labels].double().rsqrt()
        sampler = WeightedRandomSampler(weights, len(dataset), replacement=True,
                                         generator=torch.Generator().manual_seed(args.seed))
    return DataLoader(dataset, args.batch_size, shuffle=train and sampler is None, sampler=sampler,
                      num_workers=args.workers, collate_fn=collate,
                      persistent_workers=train and args.workers > 0)

def lr_factor(step, warmup, total):
    """Learning-rate multiplier: linear warmup over `warmup` steps, then cosine decay to 0 at step `total`."""
    if step < warmup:
        return (step + 1) / warmup
    return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(total - warmup, 1)))  # --epochs 1: no decay phase


def train_one_epoch(model, loader, optimizer, schedule, epoch, log_file, box_loss_weight=1.0):
    """One pass over the training set; the loss is cross-entropy plus the four Faster R-CNN losses."""
    model.train()
    start, seen = time.time(), 0
    for step, (images, labels, targets) in enumerate(loader, 1):
        targets = [{'boxes': t['boxes'].cuda(), 'labels': t['labels'].cuda()} for t in targets]
        with torch.autocast('cuda', dtype=torch.bfloat16):  # bf16 has fp32's range: no loss scaling needed
            logits, _, det_losses = model(images.cuda(), targets)
            losses = {'classification': F.cross_entropy(logits, labels.cuda()), **det_losses}
        loss = sum(value * (box_loss_weight if name == 'loss_box_reg' else 1.0)
                   for name, value in losses.items())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        schedule.step()
        seen += len(images)
        if step % 50 == 0 or step == len(loader):
            parts = ' '.join(f'{name} {value.item():.3f}' for name, value in losses.items())
            log(f'epoch {epoch} iter {step}/{len(loader)} loss {loss.item():.3f} ({parts}) '
                f'lr {optimizer.param_groups[1]["lr"]:.2e} {seen / (time.time() - start):.1f} images/s', log_file)
            start, seen = time.time(), 0


@torch.no_grad()
def evaluate(model, loader, gt_json):
    """Classification and COCO detection metrics on one split, and the detections in COCO result format."""
    model.eval()
    labels, predicted, detections, image_ids = [], [], [], []
    for images, y, targets in loader:
        with torch.autocast('cuda', dtype=torch.bfloat16):
            logits, outputs, _ = model(images.cuda())
        labels += y.tolist()
        predicted += logits.argmax(dim=1).tolist()
        for target, out in zip(targets, outputs):
            image_ids.append(target['image_id'])
            boxes = to_original_xywh(out['boxes'], target['orig_size'], geometry=target['geometry'])
            valid = (boxes[:, 2] > 0) & (boxes[:, 3] > 0)
            detections += [{'image_id': target['image_id'], 'category_id': c, 'bbox': b, 'score': s}
                           for b, c, s in zip(boxes[valid].tolist(), out['labels'][valid].tolist(),
                                             out['scores'][valid].tolist())]
    metrics = {**classification_metrics(labels, predicted), **detection_metrics(gt_json, detections, image_ids)}
    return metrics, detections


def headline(metrics):
    scores = ' '.join(f'{k} {metrics[k]:.4f}' for k in ('accuracy', 'macro_f1', 'balanced_accuracy', 'mAP', 'AP50'))
    return f'{scores} Δm {metrics["delta_m"]:+.2f}'


def validation_delta_m(metrics):
    """Mean relative change in validation macro-F1 and mAP, in percentage points."""
    return 50 * ((metrics['macro_f1'] - VAL_REFERENCE_F1) / VAL_REFERENCE_F1
                 + (metrics['mAP'] - VAL_REFERENCE_MAP) / VAL_REFERENCE_MAP)


def save_artifacts(run_dir, destination, filenames):
    """Copy completed run artifacts using atomic replacement of each destination file."""
    destination.mkdir(parents=True, exist_ok=True)
    for name in filenames:
        temporary = destination / (name + '.tmp')
        shutil.copy2(run_dir / name, temporary)
        temporary.replace(destination / name)


def main():
    args = parse_args()
    torch.manual_seed(args.seed)
    configuration = f'{args.backbone}_{args.init}_{args.adapt}'
    if args.unfreeze_last:
        configuration += f'_unfreeze{args.unfreeze_last}'
    run_dir = args.out / configuration
    run_dir.mkdir(parents=True, exist_ok=True)
    log_file = open(run_dir / 'log.txt', 'w')
    log(json.dumps(vars(args), default=str), log_file)

    # Deterministic 90/10 split of the labeled course training set.
    rows = SARMultiTask(args.data, train=False, limit=args.limit)
    n = len(rows)
    if n < 2:
        raise ValueError(f"Need at least 2 labeled images for a train/validation split; found {n}.")
    permutation = torch.randperm(n, generator=torch.Generator().manual_seed(args.seed)).tolist()
    n_val = max(1, round(n * 0.10))
    val_indices = permutation[:n_val]
    train_indices = permutation[n_val:]
    train_loader = make_loader(args, train_indices, train=True)
    val_loader = make_loader(args, val_indices, train=False)

    # Detection evaluation uses only annotations for this validation holdout.
    source_gt = json.loads((args.data / 'train' / 'instances.json').read_text())
    val_image_ids = {rows.coco_images[rows.rows[i]['file_name']]['id'] for i in val_indices}
    val_gt = {
        **source_gt,
        'images': [image for image in source_gt['images'] if image['id'] in val_image_ids],
        'annotations': [ann for ann in source_gt['annotations'] if ann['image_id'] in val_image_ids],
    }
    val_gt_path = run_dir / 'instances_val_holdout.json'
    val_gt_path.write_text(json.dumps(val_gt))
    log(f'data: {len(train_indices)} train, {len(val_indices)} validation images', log_file)

    # Model: adapters are created on the CPU, so they are added before .cuda()
    backbone = build_backbone(args.backbone, pretrained=args.init == 'pretrained', aux=args.tm_aux)  # <<< PREPROC[P3-tm-aux] (was: ...pretrained=args.init == 'pretrained'))
    if args.adapt != 'full':
        add_adapters(backbone, args.adapt)  # also freezes the backbone
    unfreeze_last_blocks(backbone, args.unfreeze_last)
    model = MultiTaskModel(backbone, task_routing=args.adapt == 'moelora', anchor_profile=args.anchor_profile)
    parameters = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f'parameters: {parameters:,} total, {trainable:,} trainable', log_file)
    if args.unfreeze_last and (parameters > 130_000_000 or trainable > 40_000_000):
        raise ValueError('Partial fine-tuning exceeds the 130M total / 40M trainable parameter budget.')
    model.cuda()

    # Optimizer and schedule: trainable backbone parameters at --backbone-lr, all new modules at --lr
    groups = parameter_groups(model, args.backbone_lr, args.lr,
                              args.unfrozen_lr if args.unfreeze_last else None)
    log('optimizer groups: ' + ', '.join(f'{group["name"]} lr {group["lr"]:.2e}' for group in groups), log_file)
    optimizer = torch.optim.AdamW(groups, lr=args.lr, weight_decay=args.weight_decay)
    warmup, total = len(train_loader), args.epochs * len(train_loader)  # one epoch of warmup
    schedule = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: lr_factor(step, warmup, total))

    # Train, validating every --eval-every epochs; the reported model is the last-epoch model
    history, start = [], time.time()
    for epoch in range(1, args.epochs + 1):
        train_one_epoch(model, train_loader, optimizer, schedule, epoch, log_file, args.box_loss_weight)
        if epoch % args.eval_every == 0 or epoch == args.epochs:
            val, val_detections = evaluate(model, val_loader, val_gt_path)
            val['delta_m'] = validation_delta_m(val)
            history.append({'epoch': epoch, **val})
            log(f'epoch {epoch} val: {headline(val)}', log_file)
    train_hours = (time.time() - start) / 3600

    # Save the trained model before final reporting.
    torch.save(model.state_dict(), run_dir / 'model.pt')
    peak_gb = torch.cuda.max_memory_allocated() / 2**30
    log(f'training {train_hours:.2f} h, peak GPU memory {peak_gb:.1f} GB', log_file)

    summary = {'args': vars(args), 'parameters': parameters, 'trainable_parameters': trainable,
               'train_hours': train_hours, 'peak_gpu_memory_gb': peak_gb,
               'val_history': history, 'val': history[-1], 'test': None}
    (run_dir / 'metrics.json').write_text(json.dumps(summary, indent=2, default=str))
    (run_dir / 'predictions_val.json').write_text(json.dumps(val_detections))
    results_dir = args.results / run_dir.name / f'seed_{args.seed}'
    save_artifacts(run_dir, results_dir, ('metrics.json', 'predictions_val.json', 'instances_val_holdout.json'))
    log(f'saved JSON results: {results_dir}', log_file)
    checkpoint_dir = args.checkpoints / run_dir.name / f'seed_{args.seed}'
    save_artifacts(run_dir, checkpoint_dir, ('model.pt', 'metrics.json'))
    log(f'saved checkpoint: {checkpoint_dir}', log_file)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/sar/make_submission.py
"""Write a submission ZIP, or predict the course validation images from a completed run.

Example: python make_submission.py --run runs/terramind-moelora-seed0/terramind_pretrained_moelora \
    --data data/course-data --out submissions/terramind_pretrained_moelora_seed0.zip
"""
import argparse
import json
import zipfile
from pathlib import Path

def write_submission(images, preds, size=512, out="submission.zip", geometry='stretch'):
    """images: from images.json; preds[file_name] = (label,
    boxes [x1, y1, x2, y2] on the model input, classes, scores).
    Both the classification label and detection classes are zero-based (0..8).
    """
    rows, dets = ["image_id,label"], []
    for im in images:
        label, boxes, classes, scores = preds[im["file_name"]]
        rows.append(f"{im['id']},{int(label)}")  # the integer id
        if geometry == 'pad':
            from sarbench.data import padded_size
            rh, rw = padded_size((im['height'], im['width']), size)
        else:
            rh, rw = size, size
        sx, sy = im["width"] / rw, im["height"] / rh
        for (x1, y1, x2, y2), c, s in zip(boxes, classes, scores):
            x1, x2 = [min(im['width'], max(0, x * sx)) for x in (x1, x2)]
            y1, y2 = [min(im['height'], max(0, y * sy)) for y in (y1, y2)]
            if x2 <= x1 or y2 <= y1:
                continue
            box = [x1, y1, x2 - x1, y2 - y1]
            dets.append({"image_id": im["id"],
                         "category_id": int(c) + 1,  # label + 1
                         "bbox": [float(v) for v in box],
                         "score": float(s)})
    with zipfile.ZipFile(out, "w") as z:
        z.writestr("classification.csv", "\n".join(rows) + "\n")
        z.writestr("detection.json", json.dumps(dets))


# >>> PREPROC[parity] rebuilt from the training args in metrics.json via the shared build_transform; gain never runs here
# (was: v2.Compose([ToImage, ResizePad or Resize((512, 512)), ToDtype(float32, scale=True), Normalize([MEAN], [STD])]))
def inference_transform(geometry='stretch', config=None):
    from sarbench.data import build_transform

    config = {'geometry': geometry} if config is None else config  # .get defaults keep old checkpoints loadable
    return build_transform(train=False, geometry=config.get('geometry', 'stretch'),
                           despeckle=config.get('despeckle', 'none'), gamma=config.get('gamma', 1.0),
                           gain=False, norm=config.get('norm', 'dataset'))
# <<< PREPROC[parity]


class SubmissionImages:
    """Unlabeled images, with the same preprocessing used during validation."""

    def __init__(self, split_root, images, geometry='stretch', config=None):  # <<< PREPROC[parity] (was: def __init__(self, split_root, images, geometry='stretch'):)
        self.root = Path(split_root)
        self.images = images
        self.transform = inference_transform(geometry, config)  # <<< PREPROC[parity] (was: inference_transform(geometry))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        from PIL import Image

        info = self.images[index]
        with Image.open(self.root / info['file_name']) as image:
            tensor = self.transform(image.convert('L'))
        return tensor, info['file_name']


def main():
    import torch
    from torch.utils.data import DataLoader
    from sarbench.adapters import add_adapters
    from sarbench.backbones import build_backbone
    from sarbench.model import MultiTaskModel

    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--run', type=Path, required=True, help='folder containing metrics.json and model.pt')
    parser.add_argument('--data', type=Path, required=True, help='course data root')
    parser.add_argument('--split', choices=['val', 'test'], default='val')
    parser.add_argument('--out', type=Path, required=True, help='submission ZIP path')
    parser.add_argument('--batch-size', type=int, default=16)
    parser.add_argument('--workers', type=int, default=8)
    args = parser.parse_args()
    config = json.loads((args.run / 'metrics.json').read_text())['args']
    split_root = args.data / args.split
    images = json.loads((split_root / 'images.json').read_text())['images']
    if not images:
        raise ValueError('The submission split contains no images.')
    if len({image['file_name'] for image in images}) != len(images):
        raise ValueError('Submission image file names must be unique.')
    if len({image['id'] for image in images}) != len(images):
        raise ValueError('Submission image IDs must be unique.')
    if args.out.exists():
        raise FileExistsError(f'Refusing to overwrite {args.out}')

    # The checkpoint contains all weights; loading it requires no pretrained download.
    backbone = build_backbone(config['backbone'], pretrained=False, aux=config.get('tm_aux', 'copy'))  # <<< PREPROC[P3-tm-aux] (was: ...pretrained=False))
    if config['adapt'] != 'full':
        add_adapters(backbone, config['adapt'])
    model = MultiTaskModel(backbone, task_routing=config['adapt'] == 'moelora',
                           anchor_profile=config.get('anchor_profile', 'default'))
    model.load_state_dict(torch.load(args.run / 'model.pt', map_location='cpu', weights_only=True))
    model.cuda().eval()
    geometry = config.get('geometry', 'stretch')
    loader = DataLoader(SubmissionImages(split_root, images, geometry, config), batch_size=args.batch_size,  # <<< PREPROC[parity] (was: SubmissionImages(split_root, images, geometry))
                        num_workers=args.workers, shuffle=False)
    predictions = {}
    with torch.inference_mode():
        for step, (batch, names) in enumerate(loader, 1):
            with torch.autocast('cuda', dtype=torch.bfloat16):
                logits, detections, _ = model(batch.cuda())
            labels = logits.argmax(dim=1).tolist()
            for name, label, detection in zip(names, labels, detections):
                # Faster R-CNN outputs category IDs 1..9; write_submission expects 0..8.
                predictions[name] = (label, detection['boxes'].cpu().tolist(),
                                     (detection['labels'] - 1).cpu().tolist(),
                                     detection['scores'].cpu().tolist())
            if step % 20 == 0 or step == len(loader):
                print(f'prediction batch {step}/{len(loader)}; {len(predictions)}/{len(images)} images', flush=True)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    temporary = args.out.with_suffix('.zip.tmp')
    write_submission(images, predictions, out=temporary, geometry=geometry)
    temporary.replace(args.out)
    print(f'Saved {args.out}: {len(images)} classification rows and '
          f'{sum(len(prediction[1]) for prediction in predictions.values())} detections', flush=True)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/sar/compute_norm_stats.py
# >>> PREPROC[norm-stats] whole file is new
"""Dataset mean/std of the model input for one --gamma / --despeckle setting, merged into norm_stats.json.

Same definition as the original constants (0.2526 / 0.2133): every pixel after resizing to 512 and scaling to
[0, 1], equal weight per pixel, here over every 3rd labeled training image.
Example: python compute_norm_stats.py --data data/course-data --gamma 0.5
"""
import argparse
import json
from pathlib import Path

import torch
from PIL import Image

from sarbench.data import SARMultiTask, build_transform


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--data', type=Path, default=Path('data/course-data'))
    p.add_argument('--gamma', type=float, default=1.0)
    p.add_argument('--despeckle', choices=['none', 'lee3'], default='none')
    # TODO(PREPROC): the stats key is gamma|despeckle only; --geometry pad includes the zero padding in the stats,
    # and a pad run would then share the key with stretch runs. The original constants were computed with stretch.
    p.add_argument('--geometry', choices=['stretch', 'pad'], default='stretch')
    p.add_argument('--limit', type=int, help='use the first N labeled images only (smoke tests)')
    p.add_argument('--out', type=Path, default=Path(__file__).resolve().parent / 'norm_stats.json')
    args = p.parse_args()

    key = f'{args.gamma:g}|{args.despeckle}'
    transform = build_transform(train=False, geometry=args.geometry, despeckle=args.despeckle,
                                gamma=args.gamma, norm='none')
    dataset = SARMultiTask(args.data, limit=args.limit)
    rows = dataset.rows[::3]
    total = total_sq = torch.zeros((), dtype=torch.float64)
    count = 0
    for i, row in enumerate(rows, 1):
        with Image.open(dataset.image_root / row['file_name']) as image:
            x = transform(image.convert('L')).double()
        total, total_sq, count = total + x.sum(), total_sq + (x * x).sum(), count + x.numel()
        if i % 500 == 0 or i == len(rows):
            print(f'{i}/{len(rows)} images', flush=True)
    mean = (total / count).item()
    std = (total_sq / count - mean ** 2).clamp_min(0).sqrt().item()
    print(f'{key}: mean {mean:.4f} std {std:.4f} over {len(rows)} images')
    if key == '1|none':
        print('Not saved: the defaults keep the original constants 0.2526 / 0.2133 so old runs stay identical.')
        return
    stats = json.loads(args.out.read_text()) if args.out.exists() else {}
    stats[key] = [mean, std]
    args.out.write_text(json.dumps(stats, indent=2))
    print(f'saved {key} to {args.out}')


if __name__ == '__main__':
    main()
# <<< PREPROC[norm-stats]

## 5. Install the pinned requirements (a few minutes)

In [ ]:
%cd /content/sar
!pip install -q -r requirements.txt
!python -c "import torch, timm, terratorch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))"

In [ ]:
!pip install -q torchaudio==2.9.0 --extra-index-url https://download.pytorch.org/whl/cu128
!python -c "import torch, torchaudio, terratorch; print(torch.__version__, torchaudio.__version__)"

In [ ]:
!pip uninstall -y torchaudio

## 6. Extract the course data

Copied from Drive to the Colab local disk, which is much faster to read during training.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import os
for sub in ('runs', 'results', 'checkpoints', 'submissions'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)
# Pretrained weights (ViT 0.4 GB, TerraMind 1.5 GB) are cached on the local disk: large downloads written
# through the Drive mount can end up truncated ("header too large"). They re-download in about a minute.
os.environ['HF_HOME'] = '/content/hf_cache'
for name in ('train.tar.gz', 'val.tar.gz'):
    path = f'{DRIVE_DATA_DIR}/{name}'
    print(('found   ' if os.path.exists(path) else 'MISSING ') + path)

In [ ]:
%cd /content/sar
os.makedirs('data/course-data', exist_ok=True)
for name in ('train.tar.gz', 'val.tar.gz'):
    print('extracting', name)
    !tar -xzf "{DRIVE_DATA_DIR}/{name}" -C data/course-data
!ls data/course-data data/course-data/train
!echo "train images: $(ls data/course-data/train/images | wc -l)"

## 6b. PREPROC normalisation stats

Only needed when `GAMMA != 1.0` or `DESPECKLE != 'none'` (with `NORM = 'dataset'`). The stats are saved to
`/content/sar/norm_stats.json` and copied to Drive, because Colab's local disk resets.
`make_submission.py` needs the same file, so run the restore cell in every new session.

In [ ]:
import os, shutil; os.path.exists(f'{DRIVE_ROOT}/norm_stats.json') and shutil.copy(f'{DRIVE_ROOT}/norm_stats.json', '/content/sar/norm_stats.json')  # PREPROC[norm-stats] restore from Drive

In [ ]:
%cd /content/sar
# >>> PREPROC[norm-stats] compute stats for the GAMMA / DESPECKLE in the settings cell (every 3rd train image, ~2-5 min)
import json, os, shutil
if NORM == 'dataset' and (GAMMA != 1.0 or DESPECKLE != 'none'):
    key = f'{GAMMA:g}|{DESPECKLE}'
    have = json.load(open('norm_stats.json')) if os.path.exists('norm_stats.json') else {}
    if key in have:
        print('already have', key, have[key])
    else:
        !python compute_norm_stats.py --data data/course-data --gamma {GAMMA} --despeckle {DESPECKLE}
        shutil.copy('norm_stats.json', f'{DRIVE_ROOT}/norm_stats.json')
else:
    print('Nothing to compute: these settings use the original constants or per-image normalisation.')
# <<< PREPROC[norm-stats]

## 7. (Optional) Quick check — a few minutes

Trains 64 images for one epoch to confirm everything works. Its scores are meaningless.

In [ ]:
%cd /content/sar
WORKERS = min(8, os.cpu_count())
!python train.py --data data/course-data --backbone vit --init pretrained --adapt lora \
  --limit 64 --epochs 1 --eval-every 1 --workers {WORKERS} \
  --out /content/smoke/runs --results /content/smoke/results --checkpoints /content/smoke/checkpoints

## 7b. PREPROC verification

Checks that the default pipeline is unchanged, that training and inference preprocess identically, and that every
new flag trains and writes a submission. The smoke runs write to `/content/smoke/preproc/`, and their norm stats go to
a separate file there, so the real `norm_stats.json` is never touched. Run the cells in section 4 again first if
the code cells were edited.

In [ ]:
%cd /content/sar
# >>> PREPROC[verify-parity] default build_transform == the original pipeline, bit for bit
import sys, torch
for m in [m for m in sys.modules if m.startswith('sarbench') or m == 'make_submission']:
    del sys.modules[m]  # pick up freshly written files
from PIL import Image
from torchvision import tv_tensors
from torchvision.transforms import v2
from sarbench.data import MEAN, STD, ResizePad, SARMultiTask, build_transform
from make_submission import inference_transform

def original_transform(train, geometry, size=512):  # copy of the old SARMultiTask.__init__ pipeline
    flips = [v2.RandomHorizontalFlip(), v2.RandomVerticalFlip()] if train else []
    return v2.Compose([v2.ToImage(), v2.ConvertBoundingBoxFormat('XYXY'),
                       ResizePad(size) if geometry == 'pad' else v2.Resize((size, size)), *flips,
                       v2.ToDtype(torch.float32, scale=True), v2.Normalize([MEAN], [STD])])

def original_inference(geometry):  # copy of the old make_submission.inference_transform
    return v2.Compose([v2.ToImage(), ResizePad() if geometry == 'pad' else v2.Resize((512, 512)),
                       v2.ToDtype(torch.float32, scale=True), v2.Normalize([MEAN], [STD])])

def raw_item(ds, i):  # exactly what SARMultiTask.__getitem__ feeds to its transform
    row = ds.rows[i]
    info = ds.coco_images[row['file_name']]
    boxes = tv_tensors.BoundingBoxes([a['bbox'] for a in ds.annotations[info['id']]], format='XYWH',
                                     canvas_size=(info['height'], info['width']), dtype=torch.float32)
    return Image.open(ds.image_root / row['file_name']).convert('L'), boxes

ds = SARMultiTask('data/course-data', limit=8)
for geometry in ('stretch', 'pad'):
    for train in (False, True):
        old, new = original_transform(train, geometry), build_transform(geometry=geometry, train=train)
        for i in range(8):
            image, boxes = raw_item(ds, i)
            torch.manual_seed(i); a = old(image, boxes)
            torch.manual_seed(i); b = new(image, boxes)
            assert torch.equal(a[0], b[0]) and torch.equal(a[1], b[1]), (geometry, train, i)
    for i in range(8):
        image, _ = raw_item(ds, i)
        assert torch.equal(original_inference(geometry)(image), inference_transform(geometry)(image)), (geometry, i)
print('PARITY OK: default training, validation and inference tensors and boxes are identical to the original')
# <<< PREPROC[verify-parity]

In [ ]:
%cd /content/sar
# >>> PREPROC[verify-train-inference] validation and submission preprocessing match with non-default flags
import sys, torch
for m in [m for m in sys.modules if m.startswith('sarbench') or m == 'make_submission']:
    del sys.modules[m]
from sarbench import data as sd
from make_submission import SubmissionImages
sd.NORM_STATS.setdefault('0.5|lee3', (0.3, 0.2))  # placeholder stats if none computed: only equality matters here
for geometry in ('stretch', 'pad'):
    config = {'geometry': geometry, 'gamma': 0.5, 'despeckle': 'lee3', 'norm': 'dataset', 'gain': True}  # gain must be ignored
    ds = sd.SARMultiTask('data/course-data', train=False, limit=4, geometry=geometry, gamma=0.5, despeckle='lee3')
    sub = SubmissionImages(ds.image_root, [{'file_name': r['file_name']} for r in ds.rows], geometry, config)
    for i in range(4):
        assert torch.equal(ds[i][0], sub[i][0]), (geometry, i)
print('TRAIN/INFERENCE PARITY OK (gamma 0.5, despeckle lee3, stretch and pad)')
# <<< PREPROC[verify-train-inference]

In [ ]:
%cd /content/sar
# >>> PREPROC[verify-smoke] 64 images x 1 epoch per flag; scores are meaningless, only completion and recorded flags matter
import glob, json, os, subprocess
WORKERS = min(8, os.cpu_count())
SMOKE_DIR = '/content/smoke/preproc'
os.makedirs(SMOKE_DIR, exist_ok=True)
SMOKE_ENV = {**os.environ, 'SARBENCH_NORM_STATS': f'{SMOKE_DIR}/norm_stats.json'}  # never the real stats file
SMOKE = {'default': '--backbone vit',
         'gamma05_gain': '--backbone vit --gamma 0.5 --gain',
         'tm_mean3': '--backbone terramind --tm-aux mean3',
         'lee3': '--backbone vit --despeckle lee3',
         'per_image': '--backbone vit --norm per_image'}

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=SMOKE_ENV)
    print('\n'.join((r.stdout + r.stderr).strip().splitlines()[-6:]))
    return r.returncode

for flags in ('--gamma 0.5', '--despeckle lee3'):  # smoke-sized stats (64 images) for the runs that need them
    sh(f'python compute_norm_stats.py --data data/course-data {flags} --limit 64 --out {SMOKE_DIR}/norm_stats.json')
status = {}
for name, flags in SMOKE.items():
    d = f'{SMOKE_DIR}/{name}'
    print(f'--- {name}: {flags}')
    status[name] = sh(f'python train.py --data data/course-data {flags} --init pretrained --adapt lora '
                      f'--limit 64 --epochs 1 --eval-every 1 --workers {WORKERS} '
                      f'--out {d}/runs --results {d}/results --checkpoints {d}/checkpoints')
print('\n=== smoke summary')
for name in SMOKE:
    paths = glob.glob(f'{SMOKE_DIR}/{name}/checkpoints/*/seed_0/metrics.json')
    recorded = {k: json.load(open(paths[0]))['args'].get(k) for k in ('gamma', 'gain', 'norm', 'despeckle', 'tm_aux')} if paths else None
    print(f'{name:13s} exit {status[name]}  metrics.json args: {recorded}')
# <<< PREPROC[verify-smoke]

In [ ]:
%cd /content/sar
# >>> PREPROC[verify-submission] make_submission.py rebuilds the preprocessing from metrics.json alone
for name in ('default', 'gamma05_gain'):
    run_dir = glob.glob(f'{SMOKE_DIR}/{name}/checkpoints/*/seed_0')[0]
    zip_path = f'{SMOKE_DIR}/{name}.zip'
    if os.path.exists(zip_path):
        os.remove(zip_path)
    sh(f'python make_submission.py --run {run_dir} --data data/course-data --split val --out {zip_path} --workers {WORKERS}')
    print(name, 'ZIP written' if os.path.exists(zip_path) else 'ZIP MISSING', zip_path)
# <<< PREPROC[verify-submission]

## 8. Train

The log, `metrics.json` and `model.pt` go to Drive under `RUN_NAME`. Each validation line prints accuracy, macro-F1, mAP, AP50 and Δm on the 939-image holdout.

In [ ]:
%cd /content/sar
WORKERS = min(8, os.cpu_count())
OUT = f'{DRIVE_ROOT}/runs/{RUN_NAME}'
RESULTS = f'{DRIVE_ROOT}/results/{GROUP}/{VARIANT}'
CHECKPOINTS = f'{DRIVE_ROOT}/checkpoints/{GROUP}/{VARIANT}'
# >>> PREPROC[run] {PREPROC_ARGS} appended to the end of the command (last line)
!python train.py --data data/course-data --backbone {BACKBONE} --init {INIT} --adapt {ADAPT} \
  --seed {SEED} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --lr {LR} --eval-every {EVAL_EVERY} \
  --workers {WORKERS} {EXTRA_ARGS} \
  --out "{OUT}" --results "{RESULTS}" --checkpoints "{CHECKPOINTS}" {PREPROC_ARGS}
# <<< PREPROC[run]

## 9. Results

In [ ]:
import glob, json
paths = sorted(glob.glob(f'{DRIVE_ROOT}/results/{RUN_NAME}/*/seed_{SEED}/metrics.json'))
if not paths:
    raise FileNotFoundError('No metrics.json found; did training finish?')
m = json.load(open(paths[-1]))
print(paths[-1])
print(f"parameters {m['parameters']:,} total, {m['trainable_parameters']:,} trainable | "
      f"{m['train_hours']:.2f} h | peak GPU {m['peak_gpu_memory_gb']:.1f} GB")
print(f"{'epoch':>5} {'acc':>7} {'F1':>7} {'mAP':>7} {'AP50':>7} {'Δm':>7}")
for h in m['val_history']:
    print(f"{h['epoch']:>5} {h['accuracy']:7.4f} {h['macro_f1']:7.4f} {h['mAP']:7.4f} {h['AP50']:7.4f} {h['delta_m']:+7.2f}")

## 10. Make the submission ZIP

Predicts the 1,426 unlabeled course validation images and saves the ZIP to Drive under `submissions/<DATE>/tests-no-submit/`. Creating it does not use a Codabench upload slot; upload it yourself.

In [ ]:
%cd /content/sar
import glob, shutil, subprocess
run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/checkpoints/{GROUP}/{VARIANT}/*/seed_{SEED}'))
if not run_dirs:
    raise FileNotFoundError('No checkpoint found; did training finish?')
RUN_DIR = run_dirs[-1]
SUB_DIR = f'{DRIVE_ROOT}/submissions/{DATE}/tests-no-submit'
os.makedirs(SUB_DIR, exist_ok=True)
ZIP = f'{SUB_DIR}/{ZIP_NAME}.zip'
shutil.copy(f'{RUN_DIR}/metrics.json', f'{SUB_DIR}/{ZIP_NAME}.metrics.json')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
with open(f'{SUB_DIR}/{ZIP_NAME}.gpu.txt', 'w') as f:
    f.write(gpu + '\n')   # one line per GPU: type and count
print('GPU:', gpu)
WORKERS = min(8, os.cpu_count())
!python make_submission.py --run "{RUN_DIR}" --data data/course-data --split val --out "{ZIP}" --workers {WORKERS}

In [ ]:
# Optional: download the ZIP to your computer
from google.colab import files
files.download(ZIP)

## 11. Download this run's files in the repo layout

Bundles every Drive file for this run into one ZIP whose paths match the git repo. Unzip it at the repo root:
`unzip -o <name>.zip -d /path/to/sar-benchmark`. The ZIP is also saved to Drive under `exports/`, because browser
downloads of large files sometimes fail.

In [ ]:
# Paths inside the ZIP are relative to DRIVE_ROOT, which mirrors the repo root:
#   results/<GROUP>/<VARIANT>/...           metrics.json, predictions_val.json, instances_val_holdout.json
#   checkpoints/<GROUP>/<VARIANT>/...       model.pt, metrics.json
#   submissions/<DATE>/tests-no-submit/<ZIP_NAME>.*   submission ZIP, .metrics.json, .gpu.txt
INCLUDE_MODEL = True   # model.pt is several hundred MB; False exports everything else

import glob, os, zipfile
from google.colab import files

patterns = [f'results/{GROUP}/{VARIANT}/**/*',
            f'checkpoints/{GROUP}/{VARIANT}/**/*',
            f'submissions/{DATE}/tests-no-submit/{ZIP_NAME}.*']
if os.path.exists(f'{DRIVE_ROOT}/norm_stats.json') and (GAMMA != 1.0 or DESPECKLE != 'none'):
    patterns.append('norm_stats.json')  # needed to rebuild gamma / despeckle preprocessing later
paths = sorted({p for pattern in patterns for p in glob.glob(f'{DRIVE_ROOT}/{pattern}', recursive=True)
                if os.path.isfile(p) and (INCLUDE_MODEL or not p.endswith('.pt'))})
if not paths:
    raise FileNotFoundError('Nothing to export; did training and the submission cell finish?')

os.makedirs(f'{DRIVE_ROOT}/exports', exist_ok=True)
EXPORT = f'{DRIVE_ROOT}/exports/{DATE}_{ZIP_NAME}_repo_files.zip'
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_STORED) as z:  # .pt and .zip are already compressed
    for path in paths:
        arcname = os.path.relpath(path, DRIVE_ROOT)
        z.write(path, arcname)
        print(f'{os.path.getsize(path) / 2**20:8.1f} MB  {arcname}')
print(f'\nSaved {EXPORT} ({os.path.getsize(EXPORT) / 2**20:.1f} MB)')
files.download(EXPORT)